#### Create reranker training data with adaptive margins

This notebook generates training data for the reranker using the newly trained retriever. It reads the existing training data, uses the retriever to identify candidate matches, and constructs training pairs from the correct answer and retrieved alternatives.

Instead of using a fixed margin for all examples, the notebook applies adaptive margins based on retriever confidence to prune the results. Examples where the retriever is less confident can tolerate a larger separation between positive and negative candidates (i.e., more neighbors will be sent to the reranker), while higher-confidence examples use a smaller margin (i.e., fewer neighbors are sent to the reranker). 

The resulting dataset is saved as Parquet and is used as input for training the reranker.

Note: this notebook assumes you have the latest version of DIBBs Env attached to your compute instance in Azure.

In [ ]:
pip install hnswlib

In [ ]:
## Imports
import os
import pickle
import torch
import io

import hnswlib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
from azureml.fsspec import AzureMachineLearningFileSystem


In [ ]:
## Global variables

# MODEL VARIABLES
MODEL_NAME = "Ellen_V2_epochs_2_batchsize_2_loss_mnrl_lr_5e-06"
TRAINED_DIR = "fine_tuned"
RERANKER_NAME = "Snowflake_snowflake-arctic-embed-l-v2.0_0.45_1e06_refined_reranker_bce_tuned_1000000_1e07"

# EMBEDDINGS VARIABLES
EMBEDDING_SIZE = 1024
EMBEDDING_NAME = "loinc_lab_names_Ellen_V2_epochs_2_batchsize_2_loss_mnrl_lr_5e-06"
EMBEDDING_FILE = f"embeddings/post_production/{EMBEDDING_NAME}_20260602"
EMBEDDING_CACHE = f"embedding_cache_retrained_{EMBEDDING_NAME}.npz"

# ANALYTIC VARIABLES
INDEX_FP = f"hnswlib_index_{MODEL_NAME.replace('/', '_')}.index"
VALIDATION_FILE = "refined_reranker_training_pairs.txt"
RERANKER_DATA_FILE = "retrained_reranker_data.parquet"

# MARGINS VARIABLES
# These are based on the prod data sent from APHL, identified in pruning.ipynb
MIN_MARGIN = 0.009
MAX_MARGIN = 0.1
LOW_SCORE = 0.7 
HIGH_SCORE = 0.95

In [ ]:
## File set up
# Authenticate to Key Vault
credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential)

# Define the workspace subscription and resources so we can instantiate a secure client
SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = 'workspaceblobstore'

# NOTE: Even though we're not directly calling any of the client functions, we do still need
# the object. Having a client instantiated acts as an authenticated connection for our compute
# instance to connect to the workspace.
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
fs = AzureMachineLearningFileSystem(
    f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
)

# Load the models
# First, check if the Retriever model exists locally--if it does, nothing to do here
if os.path.exists(MODEL_NAME):
    print("Model exists locally, loading it...")

# If there isn't a local copy, we'll fetch it from remote
else:   
    if fs.exists("models/" + TRAINED_DIR + MODEL_NAME):
        print("Found trained model, loading from remote...")
        fs.get("models/" + TRAINED_DIR + MODEL_NAME, '.')
        print("Model loaded to local memory.")
    else:
        print("Could not find model at specified path.")
        print(
            "Check model name (esp. parameter numbers, underscores, and dashes) and fetch directory."
        )

# NOTE: On better GPU, add `device="cuda"` back into the constructor
model = SentenceTransformer(MODEL_NAME)
n_params = sum(p.numel() for p in model.parameters())
print(f"Retriever model uses {n_params} parameters")


In [ ]:
## Load embedding data
if os.path.exists(EMBEDDING_CACHE):
    data = np.load(EMBEDDING_CACHE, allow_pickle=True)

    texts = data["texts"]
    vectors = data["vectors"]

    embedding_cache = dict(zip(texts, vectors))
    print("Loaded embeddings from cache")
else:
    print("Embeddings not available in cache, loading from FILE")
    # Patch torch's pickle tensor loader to force CPU
    original_load_from_bytes = torch.storage._load_from_bytes

    torch.storage._load_from_bytes = lambda b: torch.load(
        io.BytesIO(b),
        map_location=torch.device("cpu"),
        weights_only=False,
    )

    try:
        with fs.open(EMBEDDING_FILE, "rb") as fp:
            cache_data = pickle.load(fp)
    finally:
        # Restore original behavior
        torch.storage._load_from_bytes = original_load_from_bytes

    # Extract embeddings
    embeddings = np.asarray(
        cache_data["embeddings"],
        dtype=np.float32
    )

    # Extract text keys
    codes = cache_data["codes"]
    
    embedding_cache = dict(
        zip(codes, embeddings)
    )

    # Save embedding cache
    print("Saving embedding cache.")
    texts = np.array(list(embedding_cache.keys()))
    vectors = np.vstack(list(embedding_cache.values()))

    np.savez(
        EMBEDDING_CACHE,
        texts=texts,
        vectors=vectors,
    )

In [ ]:
## Load Index
# MODEL DIRECTORY
# IMPORTANT: Make sure this sub-folder is set to the correct location
# where the model's respective index file lives (or should live). For
# fine-tuned models, this should be "fine_tuned/". For TSDAE models that
# haven't been tuned, this should be "tsdae/". For all other untrained
# models, it should be "".
MODEL_SUB_DIR = "post_production/"

# ANN INDEX VARIABLES
EF_CONSTRUCTION = 400
M_VALUE = 64
EF_SEARCH = 400

# Load up or create an index over the embedding data
index = hnswlib.Index(space="cosine", dim=EMBEDDING_SIZE)

# Azure will check blob storage first using the file mount
print("Checking for cached ANN index...")
if fs.exists("indexes/" + MODEL_SUB_DIR + INDEX_FP):
    print("  Found cached index. Loading it...")

    # First, try to regularly load the index, in case we copied it here
    # from a previous run
    try:
        index.load_index(INDEX_FP)
    
    # If we can't open the file (because it's AzureML binary), then we
    # can create a local ported copy and open from that
    except:
        try:
            fs.get("indexes/" + MODEL_SUB_DIR + INDEX_FP, '.')
            index.load_index(INDEX_FP)
        
        # If that doesn't work then the file is beyond the reach of mortal
        # hands and is best left undisturbed, like all sleeping gods
        except:
            print("Could not copy or load index")
    
else:
    print("No locally cached index found. Creating hierarchical index...")
    index.init_index(
        max_elements=len(embeddings), ef_construction=EF_CONSTRUCTION, M=M_VALUE
    )
    index.add_items(embeddings, list(range(len(embeddings))))

    # Default is to save to local, working memory, so we'll need to remote copy
    # to Azure blob storage just like the reverse of copying from blob storage
    # Also clean up the local copy to avoid surplus memory charges
    index.save_index(INDEX_FP)
    fs.put(INDEX_FP, "/indexes/" + MODEL_SUB_DIR)
os.remove(INDEX_FP)

# The index should be holding approximately 276k embeddings so it better exceed 0
assert index.get_current_count() > 0
index.set_ef(EF_SEARCH)

In [ ]:
## Functions
def embed(input: str):
    """
    Embed a string
    """
    try:
        input_embedding = embedding_cache[input]
    except KeyError:
        print(f"No cached embedding found for {input}")
        input_embedding = model.encode(input, show_progress_bar=True)

    return input_embedding 

In [ ]:
## Load validation data 
correct_answers = [] 
nonstandard_inputs = []
unique_nonstandard_inputs = set()
with fs.open(VALIDATION_FILE) as fp:
    counter = 0
    for line in fp:
        # Blob storage is bytes-based, so we need to decode before string operations
        line_str = line.decode("utf-8")
        if line_str.strip() != "":
            items = line_str.strip().split("|")
            correct_answer = items[0]
            correct_answers.append(correct_answer)
            nonstandard_input = items[1]
            nonstandard_inputs.append(nonstandard_input)
            unique_nonstandard_inputs.add(nonstandard_input)

# Check that we loaded data
assert len(unique_nonstandard_inputs) > 0

In [ ]:
## Load embeddings
with fs.open(EMBEDDING_FILE) as fp:
        cache_data = pickle.load(fp)

name_codes = cache_data["codes"]

# TTC model includes four name variants but not consumer name, so there
# are just over 335k vectors that are eligible
print(f"{len(name_codes)} embeddings loaded")
assert len(name_codes) > 335_000

In [ ]:
## Embed the validation data - THIS WILL TAKE A LONG TIME THE FIRST TIME YOU RUN IT (~3 hours)

# Only encode texts missing from the embedding cache
missing = [t for t in unique_nonstandard_inputs if t not in embedding_cache]

if missing:
    for text in tqdm(missing, desc="Embedding"):
        embedding_cache[text] = model.encode(
            text,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

    # Save data 
    print("Saving updated embedding cache.")

    np.savez(
        EMBEDDING_CACHE,
        texts=np.array(list(embedding_cache.keys())),
        vectors=np.vstack(list(embedding_cache.values())),
    ) 

In [ ]:
## Pruning functions
def make_margin_func(
    low_score,
    high_score,
    max_margin,
    min_margin,
):
    """
    Create a margin function that interpolates between max_margin and min_margin
    based on the given score, low_score, and high_score. As the score (retriever confidence)
    increases, the allowable margin gets smaller.
    """

    def margin(score):
        """
        Interpolate the margin based on the score.
        """

        x = np.clip(
            (score-low_score)/(high_score-low_score),
            0,
            1,
        )

        return max_margin + x*(min_margin-max_margin)

    return margin

def create_margin_fn(min_margin, max_margin, low_score, high_score):

    return make_margin_func(
        low_score=low_score,
        high_score=high_score,
        max_margin=max_margin,
        min_margin=min_margin,
    )

def within_margin(scores, margin):
    """Determine the number of retriever candidates that are within a given margin of the top retriever score"""
    if scores is None or len(scores) == 0:
        return 0

    if isinstance(scores, np.ndarray):
        scores = scores.tolist()

    top = scores[0]

    return sum(
        top - score <= margin
        for score in scores[1:]
    )



In [ ]:
# Embed the nonstandard input and perform an approximate neighbor search with K = 10

AUTO_ACCEPT = 0.95
anchor_codes = []
positive_codes = []
negative_codes = []

margin_fn = create_margin_fn(
    min_margin=MIN_MARGIN,
    max_margin=MAX_MARGIN,
    low_score=LOW_SCORE,
    high_score=HIGH_SCORE,
)

for nonstandard_input, correct_answer in tqdm(
    zip(nonstandard_inputs, correct_answers),
    total=len(nonstandard_inputs),
    desc="Mining hard negatives",
):

    # Embed
    n_embedding = embedding_cache[nonstandard_input]

    # Note that ANN works using _distances, so we have to convert
    # to scores (cosine distance is unit-normalized to always be 
    # length-1, so we can just subtract 1 - dist)
    embedding_ids, distances = index.knn_query(n_embedding, k=10)
    hits = [
        {"corpus_id": id, "score": 1 - dist}
        for id, dist in zip(embedding_ids[0], distances[0])
    ]
    hits = sorted(hits, key=lambda x: x["score"], reverse=True)

    scores = [s["score"] for s in hits]
    texts = [t["corpus_id"] for t in hits]


    # Find neighbors at/above auto-accept threshold
    high_score_indices = [
        i for i, score in enumerate(scores)
        if score >= AUTO_ACCEPT
    ]

    # Case 1: multiple neighbors >= 0.95; keep only the >= 0.95 neighbors & do not apply the adaptive pruning
    if len(high_score_indices) > 1:

        high_score_texts = [
            texts[i]
            for i in high_score_indices
        ]

        # Remove correct answer if it is among them.
        negatives = [
            text
            for text in high_score_texts
            if text != correct_answer
        ]

        # Need at least one negative.
        if not negatives:
            continue

        anchor_codes.append(correct_answer)
        positive_codes.append(nonstandard_input)
        negative_codes.append(negatives)

        continue

    # Case 2: exactly 1 neighbor >= 0.95
    if len(high_score_indices) == 1:

        high_idx = high_score_indices[0]

        # If the only >=0.95 neighbor is the correct answer,
        # the retriever can already solve this example.
        if texts[high_idx] == correct_answer:
            continue

        # Otherwise, throw away the high-confidence incorrect
        # neighbor and continue with the remaining candidates.
        texts.pop(high_idx)
        scores.pop(high_idx)

    # Case 3: No neighbors >= 0.95 (or have finished case 2 and remove single neighbor)
     # If the correct answer was retrieved, remove it from negatives.
    if correct_answer in texts:
        correct_idx = texts.index(correct_answer)

        texts.pop(correct_idx)
        scores.pop(correct_idx)

    # Make sure there are candidates remaining.
    if not scores:
        continue

    # Adaptive margin based on remaining top score.
    top_score = scores[0]
    adaptive_margin = margin_fn(top_score)

    # Keep candidates within adaptive margin.
    num_within_margin = within_margin(
        scores,
        adaptive_margin,
    )

    pruned_texts = texts[:num_within_margin+1]

    if not pruned_texts:
        continue

    # Add to appropriate lists
    anchor_codes.append(correct_answer)
    positive_codes.append(nonstandard_input)
    negative_codes.append(pruned_texts)

# Save data
training_df = pd.DataFrame({
    "anchor_codes": anchor_codes,
    "positive_codes": positive_codes,
    "negative_codes": negative_codes,
})

training_df.to_parquet(RERANKER_DATA_FILE, index=False)
